# AI-Powered File Renamer (Ollama + gemma4:e4b)

This notebook scans a target directory for files with chosen extensions, then asks a local Ollama model - one file at a time - whether each file name should change. If the model recommends a new name, the file is renamed (or just previewed in dry-run mode).

**How to use**
1. Set the configuration in cell 2 (`TARGET_DIR`, `EXTENSIONS`, `DRY_RUN`).
2. Run the cells top to bottom.
3. Review the dry-run output, then set `DRY_RUN = False` and re-run the last code cell to apply the renames.

In [ ]:
# ------------------------- Configuration -------------------------
OLLAMA_HOST = "http://192.168.1.110:32125"
MODEL = "gemma4:e4b"        # change if your model tag differs

TARGET_DIR = "/run/user/1000/gvfs/smb-share:server=truenas.local,share=t/M"  # directory to scan
EXTENSIONS = {".cbz"}       # file extensions to consider (lowercase, include the dot)
MAX_FILES = 10               # process at most this many files (None = no limit)
DRY_RUN = True               # True = preview only, nothing is renamed

In [ ]:
def build_prompt(filename: str) -> str:
    return f"""You are a file-naming assistant for manga and doujinshi CBZ archives. Decide whether the file name below follows good naming conventions.

Naming rules:
- Title Case: capitalize the first letter of each word of the title
- the name starts with the author as [Author Name], in Title Case, without brackets in the final name
- Manga convention: [circle (Artist)] means the author is the name inside the parentheses, e.g. [toyasuaina (Toyasu Aina)] -> author is Toyasu Aina
- Other author patterns: "Author - Title", "(Author) Title", "by Author" - extract the author from them
- Only use an author that already appears in the current name - never invent one
- The manga title follows the author, preserving its meaningful words; drop series/parody names in parentheses like (Love Live! Nijigasaki High School Idol Club)
- chapter numbering goes at the very end: 001 for a single chapter, 001-005 for a chapter range (zero-padded to 3 digits); if there is no chapter number, omit it - do not add one
- remove junk tags such as [english], [digital], [unsensored], event names like (Bokura no Love Live! 34) or (C99), scanlator/group names, resolution or year tags, and similar clutter
- allowed characters: letters, digits, spaces, hyphens, and square brackets - no other special characters, no double spaces
- keep the original file extension
- if the file name is already good, respond with "KEEP" and nothing else
- if the title is in two languages and one of them is english, remove the other language and keep the english title

Target format:
Author Name Title of Manga 001.ext
Author Name Title of Manga 001-005.ext
(omit Author Name or the chapter number if not present in the current name)

Example:


RENAME
Toyasu Aina Kasumi Variable.cbz

File name: "{filename}"

Answer with EXACTLY this format:
- if the current name is fine, answer with one line: KEEP
- if the name should change, answer with exactly two lines:
RENAME
Author Name New Title 001.ext

No explanations, no extra text, no quotes."""

In [19]:
# Quick connectivity check - should print a short reply from the model
print(ask_ollama("Reply with the single word: OK"))

OK


In [ ]:
files = find_files(TARGET_DIR, EXTENSIONS)
if MAX_FILES is not None:
    files = files[:MAX_FILES]
print(f"Found {len(files)} file(s) in {Path(TARGET_DIR).expanduser().resolve()}")
print(f"Mode: {'DRY RUN - no files will be changed' if DRY_RUN else 'LIVE - files WILL be renamed'}\n")

renamed = kept = skipped = 0
for path in files:
    try:
        raw = ask_ollama(build_prompt(path.name))
    except Exception as exc:
        print(f"[ERROR]         {path.name}: {exc}")
        skipped += 1
        continue

    new_name = parse_suggestion(raw, path)
    if new_name is None or new_name == path.name:
        print(f"[KEEP]          {path.name}")
        kept += 1
        continue

    new_path = path.with_name(new_name)
    if new_path.exists():
        print(f"[SKIP]          {path.name} -> {new_name}  (target already exists)")
        skipped += 1
        continue

    if DRY_RUN:
        print(f"[WOULD RENAME]  {path.name} -> {new_name}")
    else:
        path.rename(new_path)
        print(f"[RENAMED]       {path.name} -> {new_name}")
    renamed += 1

print(f"\nDone. renamed: {renamed} | kept: {kept} | skipped: {skipped}")